# End-to-End GNN-BERT Music Context Demo

**Course:** CSE425 / EEE474 / CSE715 — Neural Networks  
**Project:** GNN-Based BERT for Understanding Context from Music  
**Prepared By:** Tanisha Islam

This notebook demonstrates one complete multimodal forward-pass example using:

1. A real preprocessed MusicCaps audio graph
2. A genuine MusicCaps-style text caption
3. DistilBERT text encoding
4. GNN audio-graph encoding
5. GNN-BERT cross-attention fusion
6. Multi-label music-tag prediction
7. Cross-attention visualization

The graph preprocessing is performed by the main project pipeline. A representative
processed graph is included in `data/examples/` so the demo can be inspected
without downloading the complete audio dataset.

**Note:** The repository does not include large trained model checkpoints.
Therefore, this notebook demonstrates the complete inference pipeline and model
forward pass rather than claiming that the displayed probabilities are final
trained-test predictions. Final experimental results are stored in
`results/metrics.json`.


In [ ]:
import os
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt

# Allow imports whether notebook is launched from project root or notebooks/
project_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, project_root)

from src.bert_encoder import MusicBertEncoder
from src.fusion_model import GNNBertFusionModel
from src.prepare_real_dataset import TOP_TAGS as TAGS_VOCAB

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)

print("Device:", device)
print("Number of tags:", len(TAGS_VOCAB))


## 1. Load a Real Processed Music Graph

The repository includes representative graph examples generated from the real
MusicCaps preprocessing pipeline.


In [ ]:
graph_path = os.path.join(project_root, "data", "examples", "mc_001_graph.json")

with open(graph_path, "r", encoding="utf-8") as f:
    graph = json.load(f)

x = torch.tensor(graph["x"], dtype=torch.float32, device=device)
edge_index = torch.tensor(graph["edge_index"], dtype=torch.long, device=device)
edge_weight = torch.tensor(graph["edge_weight"], dtype=torch.float32, device=device)

print("Graph:", graph_path)
print("Node feature shape:", tuple(x.shape))
print("Edge index shape:", tuple(edge_index.shape))
print("Edge weight shape:", tuple(edge_weight.shape))


## 2. Text Context

A text caption is paired with the audio graph and encoded with pretrained
DistilBERT.


In [ ]:
caption = (
    "The low quality recording features a ballad song that contains "
    "sustained strings, mellow piano melody and soft female vocal singing over it."
)

print("Caption:")
print(caption)


## 3. Initialize the GNN-BERT Cross-Attention Model


In [ ]:
model = GNNBertFusionModel(
    num_tags=len(TAGS_VOCAB),
    in_node_dim=32,
    gnn_hidden_dim=128,
    gnn_out_dim=128,
    fusion_type="cross_attention"
).to(device)

bert_encoder = MusicBertEncoder()
model.eval()

print("GNN-BERT cross-attention model initialized.")


## 4. Tokenize Text and Run Multimodal Inference


In [ ]:
tokens = bert_encoder.tokenize([caption], device=device)

input_ids = tokens["input_ids"]
attention_mask = tokens.get("attention_mask")

with torch.no_grad():
    output = model(
        x,
        edge_index,
        input_ids,
        attention_mask=attention_mask,
        edge_weight=edge_weight
    )

tag_probs = output["tag_probs"].squeeze(0).detach().cpu().numpy()

top_indices = np.argsort(-tag_probs)[:5]

print("Top-5 output tags from the forward pass:")
for rank, idx in enumerate(top_indices, 1):
    print(
        f"{rank}. {TAGS_VOCAB[idx]:<15} "
        f"Probability: {tag_probs[idx]:.4f}"
    )


## 5. Inspect Multimodal Representations


In [ ]:
print("Graph embedding shape:", tuple(output["g"].shape))
print("Text embedding shape:", tuple(output["t_cls"].shape))
print("Fused representation shape:", tuple(output["z"].shape))
print("Tag probability shape:", tuple(output["tag_probs"].shape))


## 6. Cross-Attention Visualization

The following plot shows how the graph representation distributes attention
across BERT tokens for this example.


In [ ]:
att = output["att_weights"].detach().cpu().squeeze()

if att.ndim == 2:
    token_attention = att.mean(dim=0).numpy()
elif att.ndim == 1:
    token_attention = att.numpy()
else:
    token_attention = att.reshape(-1).numpy()

token_strings = bert_encoder.tokenizer.convert_ids_to_tokens(
    input_ids[0].detach().cpu().tolist()
)

n = min(len(token_strings), len(token_attention))
token_strings = token_strings[:n]
token_attention = token_attention[:n]

plt.figure(figsize=(12, 4))
plt.bar(range(n), token_attention)
plt.xticks(range(n), token_strings, rotation=60, ha="right")
plt.ylabel("Attention Weight")
plt.xlabel("DistilBERT Tokens")
plt.title("GNN-BERT Cross-Attention for One MusicCaps Example")
plt.tight_layout()
plt.show()


## Demo Summary

This notebook demonstrates the complete multimodal model path:

**Processed music graph → GNN → DistilBERT caption encoding → cross-attention fusion → tag probabilities**

The complete dataset preprocessing and model-training pipeline is implemented in
`src/prepare_real_dataset.py` and `src/train.py`.

Final measured experimental results are available in:

`results/metrics.json`
